# Использование ML-моделей для классификации

Таблица результатов ML-моделей
|Модель|Гиперпараметры|f1|ROC-AUC|accuracy|
|:----:|:----:|:----:|:----:|:----:|
|SVC|gamma = 0.001, C = 0.30|0.63|0.79|0.72|
|SVC|gamma = 'scale', C = 1.0|**0.63**|**0.79**|**0.74**|
|RandomForest|n_estimators = 200, min_samples_split = 10, min_samples_leaf = 4,max_depth = 5|0.53|0.72|0.69|
|Logistic Regression|solver = liblinear, C = 0.0|0.60|0.75|0.70|
|LightGBM|learning_rate = 0.03, min_child_samples = 43, n_estimators = 140,num_leaves = 47|0.59|0.76|0.73|
|CatBoost|learning_rate = 0.1, l2_leaf_reg = 5, iterations = 100, depth = 6|0.54|0.75|0.74|

In [2]:
!pip install catboost lightgbm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.4 MB/s eta 0:00:00


In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, balanced_accuracy_score, classification_report, confusion_matrix
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
import joblib
import time
from tqdm import tqdm
import gdown
import zipfile
from scipy.stats import uniform, loguniform, randint
import warnings
warnings.filterwarnings('ignore')

In [4]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Загрузка датасета

In [5]:
FILE_ID = "12XMzJ1xRFXAzX8oZ_XwcF-0j9J74ZXNB"
ZIP_PATH = "dataset.zip"
EXTRACT_PATH = "data"
try:
    gdown.download(
        id=FILE_ID,
        output=ZIP_PATH,
        quiet=False,
        fuzzy=True
    )
    print("Загрузка архива завершена!")
except Exception as e:
    print(f"Ошибка при загрузке: {e}")

Downloading...
From (original): https://drive.google.com/uc?id=12XMzJ1xRFXAzX8oZ_XwcF-0j9J74ZXNB
From (redirected): https://drive.google.com/uc?id=12XMzJ1xRFXAzX8oZ_XwcF-0j9J74ZXNB&confirm=t&uuid=e0003c1e-2732-4399-bfda-3b223a05cfc0
To: /content/dataset.zip
100%|██████████| 695M/695M [00:23<00:00, 29.6MB/s]

Загрузка архива завершена!


In [6]:
if os.path.exists(ZIP_PATH):
    print("Распаковка архива")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall(EXTRACT_PATH)

Распаковка архива


In [7]:
good_dir = "data/хорошо"
bad_dir = "data/плохо"

In [8]:
file_data = []
for label, folder in [(0, good_dir), (1, bad_dir)]:
    for fname in os.listdir(folder):
        if fname.endswith('.wav'):
            path = os.path.join(folder, fname)
            twister = fname.split('__')[1].replace('.wav', '')
            file_data.append([path, label, fname, twister])

df = pd.DataFrame(file_data, columns=['path', 'label', 'filename', 'twister'])

In [8]:
df

,path,label,filename,twister
0,data/хорошо/eu.4feaa83c-dd68-4a69-9d13-db1ae1b...,0,eu.4feaa83c-dd68-4a69-9d13-db1ae1b3a270__тлщ.wav,тлщ
1,data/хорошо/eu.5e2b58f0-8b68-4a53-a65c-9ec7441...,0,eu.5e2b58f0-8b68-4a53-a65c-9ec74416abe9__срл.wav,срл
2,data/хорошо/eu.63784c86-1d20-4bfa-aee4-5a77a9c...,0,eu.63784c86-1d20-4bfa-aee4-5a77a9cdf458__трл.wav,трл
3,data/хорошо/eu.d006f83b-a3a1-4a63-83fc-0e2fbe0...,0,eu.d006f83b-a3a1-4a63-83fc-0e2fbe0273be__трлш.wav,трлш
4,data/хорошо/eu.f31a3a75-aba9-4a45-bdd7-8a6c592...,0,eu.f31a3a75-aba9-4a45-bdd7-8a6c592d49b4__стрл.wav,стрл
...,...,...,...,...
2768,data/плохо/eu.7eeb191c-2c31-48cf-9487-a1bf63c2...,1,eu.7eeb191c-2c31-48cf-9487-a1bf63c2a033__срл.wav,срл
2769,data/плохо/eu.851a6d96-3dfd-40fc-986a-5a110917...,1,eu.851a6d96-3dfd-40fc-986a-5a1109174633__тл.wav,тл
2770,data/плохо/eu.c81f65c4-7e6e-49ed-8e21-559a4d8d...,1,eu.c81f65c4-7e6e-49ed-8e21-559a4d8d4d47__стрлш...,стрлш
2771,data/плохо/eu.7163b349-3aa2-45fb-ac80-2091f719...,1,eu.7163b349-3aa2-45fb-ac80-2091f7192d1e__чстрл...,чстрл


## Извлечение MFCC - признаков

Берём из EDA окно длительностью 10 сек

In [9]:
TARGET_SR = 16000
MAX_LEN = 10 * TARGET_SR
def load_audio(path):
    y, sr = librosa.load(path, sr=TARGET_SR)
    if len(y) > MAX_LEN:
        start = (len(y) - MAX_LEN) // 2
        y = y[start:start+MAX_LEN]
    else:
        pad_len = MAX_LEN - len(y)
        y = np.pad(y, (pad_len//2, pad_len - pad_len//2), mode='constant')
    return y

In [10]:
def extract_features(y):
    mfcc = librosa.feature.mfcc(y=y, sr=TARGET_SR, n_mfcc=40)
    mfcc_mean = np.mean(mfcc, axis=1)
    mfcc_std = np.std(mfcc, axis=1)
    mfcc_delta = librosa.feature.delta(mfcc)
    mfcc_delta2 = librosa.feature.delta(mfcc, order=2)
    delta_mean = np.mean(mfcc_delta, axis=1)
    delta_std = np.std(mfcc_delta, axis=1)
    delta2_mean = np.mean(mfcc_delta2, axis=1)
    delta2_std = np.std(mfcc_delta2, axis=1)
    cent = librosa.feature.spectral_centroid(y=y, sr=TARGET_SR)
    cent_mean, cent_std = np.mean(cent), np.std(cent)
    bw = librosa.feature.spectral_bandwidth(y=y, sr=TARGET_SR)
    bw_mean, bw_std = np.mean(bw), np.std(bw)
    contrast = librosa.feature.spectral_contrast(y=y, sr=TARGET_SR)
    contrast_mean = np.mean(contrast, axis=1)
    zcr = librosa.feature.zero_crossing_rate(y)
    zcr_mean, zcr_std = np.mean(zcr), np.std(zcr)
    rms = librosa.feature.rms(y=y)
    rms_mean, rms_std = np.mean(rms), np.std(rms)
    features = np.concatenate([
        mfcc_mean, mfcc_std,
        delta_mean, delta_std,
        delta2_mean, delta2_std,
        [cent_mean, cent_std, bw_mean, bw_std],
        contrast_mean,
        [zcr_mean, zcr_std, rms_mean, rms_std]
    ])
    return features

In [11]:
X_list = []
y_list = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Извлечение признаков"):
    audio = load_audio(row['path'])
    feats = extract_features(audio)
    X_list.append(feats)
    y_list.append(row['label'])
X = np.array(X_list)
y = np.array(y_list)
np.save('X.npy', X)
np.save('y.npy', y)

Извлечение признаков: 100%|██████████| 2773/2773 [06:22<00:00,  7.25it/s]


In [12]:
X.shape, y.shape

((2773, 255), (2773,))

## Разделение данных на выборки и обучение baseline - модели

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=RANDOM_STATE, stratify=y_train
)

In [14]:
X_train.shape, X_val.shape, X_test.shape

((1774, 255), (444, 255), (555, 255))

In [13]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

В качестве базового решения берём стандратную SVM - модель, так как в других исследованиях по проблеме классификации дефектов речи именно она в связке с MfCC-признаками показала лучший результат.

In [16]:
svm_baseline = SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE, class_weight='balanced')
start = time.time()
svm_baseline.fit(X_train_scaled, y_train)
fit_time = time.time() - start
y_pred = svm_baseline.predict(X_test_scaled)
y_proba = svm_baseline.predict_proba(X_test_scaled)[:, 1]

In [17]:
print(f"""
    'Accuracy': {accuracy_score(y_test, y_pred)},
    'F1': {f1_score(y_test, y_pred)},
    'ROC-AUC': {roc_auc_score(y_test, y_proba)},
    'Balanced Acc': {balanced_accuracy_score(y_test, y_pred)},
    'Время (с)': {fit_time}
""")


    'Accuracy': 0.7387387387387387,
    'F1': 0.629156010230179,
    'ROC-AUC': 0.7908518518518519,
    'Balanced Acc': 0.7243333333333333,
    'Время (с)': 4.183000802993774



In [18]:
print(classification_report(y_test, y_pred, target_names=['Хорошо', 'Плохо']))

              precision    recall  f1-score   support

      Хорошо       0.83      0.77      0.80       375
       Плохо       0.58      0.68      0.63       180

    accuracy                           0.74       555
   macro avg       0.71      0.72      0.71       555
weighted avg       0.75      0.74      0.74       555



## Подбор гиперпараметров для Logistic Regression

In [20]:
param_lr = {
    'C': np.logspace(-3, 2, 20),
    'solver': ['liblinear', 'lbfgs']
}
lr = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE, class_weight='balanced')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rs_lr = RandomizedSearchCV(lr, param_lr, n_iter=15, cv=cv, scoring='f1', random_state=RANDOM_STATE, n_jobs=-1)

In [ ]:
rs_lr.fit(X_train_scaled, y_train)
best_lr = rs_lr.best_estimator_

In [ ]:
rs_lr.best_params_

{'solver': 'liblinear', 'C': np.float64(0.003359818286283781)}

In [ ]:
y_pred_lr = best_lr.predict(X_test_scaled)
y_proba_lr = best_lr.predict_proba(X_test_scaled)[:, 1]
print(f"""
    'Accuracy': {accuracy_score(y_test, y_pred_lr)},
    'F1': {f1_score(y_test, y_pred_lr)},
    'ROC-AUC': {roc_auc_score(y_test, y_proba_lr)},
    'Balanced Acc': {balanced_accuracy_score(y_test, y_pred_lr)}
""")


    'Accuracy': 0.7009009009009008,
    'F1': 0.5951219512195122,
    'ROC-AUC': 0.7536550576488767,
    'Balanced Acc': 0.6958353143943896



## Подбор гиперпараметров для Random Forest

In [ ]:
param_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [2, 5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}
rf = RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced', n_jobs=-1)
rs_rf = RandomizedSearchCV(rf, param_rf, n_iter=20, cv=cv, scoring='f1', random_state=RANDOM_STATE, n_jobs=-1)

In [ ]:
rs_rf.fit(X_train_scaled, y_train)
best_rf = rs_rf.best_estimator_

In [ ]:
rs_rf.best_params_

{'n_estimators': 200,
 'min_samples_split': 10,
 'min_samples_leaf': 4,
 'max_features': 'sqrt',
 'max_depth': 5}

In [ ]:
y_pred_rf = best_rf.predict(X_test_scaled)
y_proba_rf = best_rf.predict_proba(X_test_scaled)[:, 1]
print(f"""
    'Accuracy': {accuracy_score(y_test, y_pred_rf)},
    'F1': {f1_score(y_test, y_pred_rf)},
    'ROC-AUC': {roc_auc_score(y_test, y_proba_rf)},
    'Balanced Acc': {balanced_accuracy_score(y_test, y_pred_rf)}
""")


    'Accuracy': 0.6918918918918919,
    'F1': 0.5340599455040872,
    'ROC-AUC': 0.7240431475098063,
    'Balanced Acc': 0.6540621656959467



## Подбор гиперпараметров для SVM

In [21]:
param_svm = {
    'C': np.logspace(-2, 2, 20),
    'gamma': np.logspace(-3, 0, 20)
}
svm = SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE, class_weight='balanced')
rs_svm = RandomizedSearchCV(svm, param_svm, n_iter=50, cv=cv, scoring='f1', random_state=RANDOM_STATE, n_jobs=-1, verbose=1)

In [22]:
rs_svm.fit(X_train_scaled, y_train)
best_svm = rs_svm.best_estimator_

Fitting 5 folds for each of 50 candidates, totalling 250 fits


In [23]:
rs_svm.best_params_

{'gamma': np.float64(0.0014384498882876629),
 'C': np.float64(0.29763514416313175)}

In [24]:
y_pred_svm = best_svm.predict(X_test_scaled)
y_proba_svm = best_svm.predict_proba(X_test_scaled)[:, 1]
print(f"""
    'Accuracy': {accuracy_score(y_test, y_pred_svm)},
    'F1': {f1_score(y_test, y_pred_svm)},
    'ROC-AUC': {roc_auc_score(y_test, y_proba_svm)},
    'Balanced Acc': {balanced_accuracy_score(y_test, y_pred_svm)}
""")


    'Accuracy': 0.7189189189189189,
    'F1': 0.6320754716981132,
    'ROC-AUC': 0.7944000000000001,
    'Balanced Acc': 0.7255555555555555



Незначительно увеличилась f1, но упала accuracy. Оставляем первую обученную SVM как лучшее решение для ML-моделей на данном этапе

## Подбор гиперпараметров для LightGBM

In [ ]:
param_lgb = {
    'num_leaves': randint(20, 150),
    'learning_rate': uniform(0.01, 0.3),
    'n_estimators': randint(100, 500),
    'min_child_samples': randint(5, 50)
}
lgb = LGBMClassifier(random_state=RANDOM_STATE, class_weight='balanced', n_jobs=-1, verbose=-1)
rs_lgb = RandomizedSearchCV(lgb, param_lgb, n_iter=25, cv=cv, scoring='f1', random_state=RANDOM_STATE, n_jobs=-1)

In [ ]:
rs_lgb.fit(X_train_scaled, y_train)
best_lgb = rs_lgb.best_estimator_

In [ ]:
rs_lgb.best_params_

{'learning_rate': np.float64(0.03221339552022711),
 'min_child_samples': 43,
 'n_estimators': 140,
 'num_leaves': 47}

In [ ]:
y_pred_lgb = best_lgb.predict(X_test_scaled)
y_proba_lgb = best_lgb.predict_proba(X_test_scaled)[:, 1]
print(f"""
    'Accuracy': {accuracy_score(y_test, y_pred_lgb)},
    'F1': {f1_score(y_test, y_pred_lgb)},
    'ROC-AUC': {roc_auc_score(y_test, y_proba_lgb)},
    'Balanced Acc': {balanced_accuracy_score(y_test, y_pred_lgb)}
""")


    'Accuracy': 0.7279279279279279,
    'F1': 0.5907859078590786,
    'ROC-AUC': 0.7590930702484251,
    'Balanced Acc': 0.69675650778557



## Подбор гиперпараметров для CatBoost

In [ ]:
param_cat = {
    'depth': [4, 6],
    'learning_rate': [0.03, 0.1],
    'iterations': [100, 200],
    'l2_leaf_reg': [3, 5, 7]
}

In [ ]:
cat = CatBoostClassifier(random_seed=RANDOM_STATE, verbose=0, task_type='CPU')
rs_cat = RandomizedSearchCV(cat, param_cat, n_iter=15, cv=cv, scoring='f1', random_state=RANDOM_STATE, n_jobs=-1)

In [ ]:
rs_cat.fit(X_train_scaled, y_train)
best_cat = rs_cat.best_estimator_

In [ ]:
rs_cat.best_params_

{'learning_rate': 0.1, 'l2_leaf_reg': 5, 'iterations': 100, 'depth': 6}

In [ ]:
y_pred_cat = best_cat.predict(X_test_scaled)
y_proba_cat = best_cat.predict_proba(X_test_scaled)[:, 1]
print(f"""
    'Accuracy': {accuracy_score(y_test, y_pred_cat)},
    'F1': {f1_score(y_test, y_pred_cat)},
    'ROC-AUC': {roc_auc_score(y_test, y_proba_cat)},
    'Balanced Acc': {balanced_accuracy_score(y_test, y_pred_cat)}
""")


    'Accuracy': 0.7441441441441441,
    'F1': 0.5448717948717948,
    'ROC-AUC': 0.7518572447402828,
    'Balanced Acc': 0.6736003803637228



## Обучение лучшей модели на полной тренировочной выборке

In [21]:
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

In [22]:
X_train.shape, X_test.shape

((2218, 255), (555, 255))

In [23]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [24]:
svm_best = SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE, class_weight='balanced')
svm_best.fit(X_train_scaled, y_train)
y_pred = svm_best.predict(X_test_scaled)
y_proba = svm_best.predict_proba(X_test_scaled)[:, 1]

In [25]:
print(f"""
    'Accuracy': {accuracy_score(y_test, y_pred)},
    'F1': {f1_score(y_test, y_pred)},
    'ROC-AUC': {roc_auc_score(y_test, y_proba)},
    'Balanced Acc': {balanced_accuracy_score(y_test, y_pred)}
""")


    'Accuracy': 0.7405405405405405,
    'F1': 0.6210526315789474,
    'ROC-AUC': 0.7965333333333332,
    'Balanced Acc': 0.7184444444444444



In [53]:
print(classification_report(y_test, y_pred, target_names=['Хорошо', 'Плохо']))

              precision    recall  f1-score   support

      Хорошо       0.83      0.78      0.80       375
       Плохо       0.59      0.66      0.62       180

    accuracy                           0.74       555
   macro avg       0.71      0.72      0.71       555
weighted avg       0.75      0.74      0.74       555



## Анализ дефектов и выбор скороговорок

In [47]:
test_df = df.loc[idx_test].copy()
test_df['predicted'] = y_pred
test_df['true_label'] = y_test
test_df['correct'] = (test_df['predicted'] == test_df['true_label'])

In [67]:
twister_stats = test_df.groupby('twister').agg({
    'correct': ['mean', 'count'],
    'true_label': 'mean'
}).round(3)

twister_stats.columns = ['accuracy', 'count', 'bad_ratio']

In [68]:
twister_stats[(twister_stats['accuracy'] >= 0.7) & (twister_stats['count'] >= 16)]

,accuracy,count,bad_ratio
twister,,,
рлш,0.850,20,0.300
стр,0.833,24,0.333
стрлц,0.737,19,0.158
стрлш,0.714,28,0.321
тл,0.824,17,0.176
тлщ,0.824,17,0.176
трлш,0.706,17,0.647
трш,0.737,19,0.421
чстрл,0.875,16,0.250


In [54]:
joblib.dump(svm_best, 'svm_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

['scaler.pkl']

## Вывод
- Наилучшие результаты из Ml-моделей дает SVC с гиперпараметрами C=1.0, f1 = 0.63, ROC-AUC = 0.79, accuracy = 0.74
- Были выбраны 5 скороговорок для пользователя с учётом accuracy, количества скороговорок данного типа и соотношения "плохих" и "хороших" скороговорок в данном типе.
    - "рлш" - "Наш голова вашего голову головой переголовил, перевыголовил"  
    - "стр" - "Хитрую сороку поймать морока, а сорок сорок — сорок морок"     
    - "трш" - "Шарики шарикоподшипника шарят по подшипнику"
    - "стрлш" - "Токарь Раппопорт пропил пропуск, рашпиль и суппорт"
    - "чстрл" - "Четыре чёрненьких чумазеньких чертёнка чертили чёрными чернилами чертёж чрезвычайно чисто"
- Скачаны веса модели